# funcgenomic walkthrough

A short tour of the prototype. Three things in order:

1. Load the curated atlas and the effector edge table.
2. Score every row, then roll up to host processes.
3. Look at the cross class convergence as a heatmap and a bipartite network.

This notebook intentionally avoids any heavy dependency. Only stdlib plus matplotlib for the plots.

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO / 'src'))

from funcgenomic.scoring import default_csv_path, load_targets, rank
from funcgenomic.atlas import rollup_processes
from funcgenomic.novelty import default_edges_path, load_edges, signal

print('repo root:', REPO)
print('atlas csv:', default_csv_path())
print('edges csv:', default_edges_path())

## 1. Load the atlas

Each row is one crop x pathogen x host target line. The convergence score is computed at load time using the weights in `configs/weights.toml` (defaults baked into `scoring.DEFAULT_WEIGHTS`).

In [ ]:
rows = load_targets(default_csv_path())
print(f'loaded {len(rows)} rows')
for r in rank(rows)[:5]:
    print(f'{r.crop:<10} {r.pathogen_class:<10} {r.host_process:<32} score={r.convergence_score:.2f}')

## 2. Roll up to host processes

Per row scoring is useful for inspection. The actual decision unit is the host process. The rollup combines rows that share a process and uses the edge table to decide breadth across pathogen classes.

In [ ]:
from collections import defaultdict

edges = load_edges()
classes_by_module = defaultdict(set)
for e in edges:
    classes_by_module[e['host_module']].add(e['pathogen_class'])
novelty_modules = {s.host_module for s in signal(edges, min_classes=3) if s.flagged}

processes = rollup_processes(rows, edge_classes_by_module=classes_by_module, novelty_modules=novelty_modules)
for p in processes:
    print(f'{p.host_process:<34} breadth={p.breadth_classes}  evidence={p.evidence_score:.2f}  novelty={p.novelty_flag}  score={p.convergence_score:.2f}')

## 3. Visualise the rollup as a bar chart

The score itself is a stack of contributions. The chart below sorts host processes by total score and colours the bar by whether the structural novelty signal is on.

In [ ]:
import matplotlib.pyplot as plt

names = [p.host_process for p in processes]
scores = [p.convergence_score for p in processes]
colors = ['#e74c3c' if p.novelty_flag else '#2980b9' for p in processes]

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(names[::-1], scores[::-1], color=colors[::-1])
ax.set_xlabel('convergence score')
ax.set_title('host processes ranked')
ax.grid(axis='x', linestyle=':', alpha=0.4)
for i, p in enumerate(processes[::-1]):
    ax.text(p.convergence_score + 0.02, i, f'b={p.breadth_classes}', va='center', fontsize=8)
fig.tight_layout()
plt.show()

Red bars are the structurally promiscuous modules (three or more pathogen classes documented at the same host node). The number to the right of each bar is the breadth in pathogen classes.

## 4. Cross class convergence heatmap

Rows are host modules. Columns are pathogen classes. A filled cell means at least one documented exploit edge in the data.

In [ ]:
import numpy as np

all_classes = sorted({e['pathogen_class'] for e in edges})
modules = sorted(classes_by_module.keys())
matrix = np.zeros((len(modules), len(all_classes)))
for i, m in enumerate(modules):
    for j, c in enumerate(all_classes):
        if c in classes_by_module[m]:
            matrix[i, j] = 1

fig, ax = plt.subplots(figsize=(6, 4))
ax.imshow(matrix, aspect='auto', cmap='Blues')
ax.set_xticks(range(len(all_classes)))
ax.set_xticklabels(all_classes, rotation=30, ha='right')
ax.set_yticks(range(len(modules)))
ax.set_yticklabels(modules)
ax.set_title('cross class convergence (atlas + edges)')
fig.tight_layout()
plt.show()

## 5. Bipartite network

Pathogens on the left, host modules on the right. Edges are documented exploit relationships. Hubs (SWEET, DMR6, coreceptor) are the broad spectrum candidates.

In [ ]:
pathogens = sorted({e['pathogen'] for e in edges})
host_modules = sorted({e['host_module'] for e in edges})
class_color = {
    'bacterium': '#e74c3c',
    'fungus': '#27ae60',
    'oomycete': '#2980b9',
    'virus': '#8e44ad',
    'protist': '#f39c12',
}
pathogen_class = {e['pathogen']: e['pathogen_class'] for e in edges}

fig, ax = plt.subplots(figsize=(10, 7))
y_left = {p: i for i, p in enumerate(pathogens)}
y_right = {m: i * (len(pathogens) / max(len(host_modules) - 1, 1)) for i, m in enumerate(host_modules)}

for e in edges:
    x = [0, 1]
    y = [y_left[e['pathogen']], y_right[e['host_module']]]
    ax.plot(x, y, color=class_color.get(e['pathogen_class'], '#888'), alpha=0.55, linewidth=1.2)

for p, y in y_left.items():
    ax.scatter(0, y, color=class_color.get(pathogen_class.get(p, ''), '#888'), s=60, zorder=3)
    ax.text(-0.02, y, p, ha='right', va='center', fontsize=8)
for m, y in y_right.items():
    ax.scatter(1, y, color='#222', s=80, zorder=3)
    ax.text(1.02, y, m, ha='left', va='center', fontsize=9)

ax.set_xlim(-0.6, 1.6)
ax.set_xticks([])
ax.set_yticks([])
ax.set_title('pathogen to host module bipartite network')
for spine in ax.spines.values():
    spine.set_visible(False)
fig.tight_layout()
plt.show()

## 6. Try a different weighting

The whole point of the rule is that the weights are visible. Below we shift the rule to favour deployability and tractability over breadth, and watch the ranking move.

In [ ]:
alt_weights = {
    'evidence': 0.20,
    'breadth': 0.15,
    'tractability': 0.30,
    'deployability': 0.25,
    'fitness_risk': 0.10,
}
alt_rows = load_targets(default_csv_path(), weights=alt_weights)
alt_processes = rollup_processes(alt_rows, weights=alt_weights, edge_classes_by_module=classes_by_module, novelty_modules=novelty_modules)
for p in alt_processes:
    print(f'{p.host_process:<34} score={p.convergence_score:.2f}')

## 7. Recap

What this notebook actually showed:

- The atlas is small but structured. 14 rows, 9 host processes, 23 effector edges.
- The ranking is dominated by evidence and breadth, with novelty as a small tiebreaker.
- The convergence pattern is real in the data we have. SWEET, DMR6, and the coreceptor module reach three pathogen classes in the edge table.
- Changing the weights changes the ordering. That is a feature, not a bug. Decision support, not oracle.